# Model 2 - CNN Parallel
Notebook train và test CNN Parallel.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from model2.models import build_models

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / 'data').exists() and PROJECT_DIR != PROJECT_DIR.parent: PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data' / 'VN_trash_classification_preprocessing'
EPOCHS, BATCH_SIZE, LR = 30, 32, 1e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
normalize = transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
train_tf = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ColorJitter(.1,.1,.1), transforms.ToTensor(), normalize])
test_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), normalize])
train_ds = ImageFolder(DATA_DIR/'train', transform=train_tf); val_ds = ImageFolder(DATA_DIR/'val', transform=test_tf); test_ds = ImageFolder(DATA_DIR/'test', transform=test_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True); val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE); test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print('Train:', len(train_ds), '| Validation:', len(val_ds), '| Test:', len(test_ds))

In [ ]:
# Tạo model, loss và optimizer
model = build_models(len(class_names))['cnn_parallel'].to(device)
counts = Counter(train_ds.targets)
weights = torch.tensor([len(train_ds)/(len(class_names)*counts[i]) for i in range(len(class_names))], dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*.01)
print(model)

In [ ]:
# Train và validation
def run_epoch(loader, training=False):
    model.train(training); total_loss = correct = seen = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            outputs = model(images); loss = loss_fn(outputs, labels)
            if training: loss.backward(); optimizer.step()
        total_loss += loss.item()*labels.size(0); correct += (outputs.argmax(1)==labels).sum().item(); seen += labels.size(0)
    return total_loss/seen, correct/seen
best_acc, best_state = 0, None
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True); val_loss, val_acc = run_epoch(val_loader); scheduler.step()
    if val_acc > best_acc: best_acc, best_state = val_acc, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.2%} | Val: {val_acc:.2%}')
model.load_state_dict(best_state); checkpoint = PROJECT_DIR/'src/model2/checkpoints/cnn_parallel_notebook.pt'; checkpoint.parent.mkdir(parents=True, exist_ok=True); torch.save(model.state_dict(), checkpoint)
print('Best validation accuracy:', f'{best_acc:.2%}', '| Checkpoint:', checkpoint)

In [ ]:
# Test metrics
model.eval(); true_labels, predicted_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device)); true_labels += labels.tolist(); predicted_labels += outputs.argmax(1).cpu().tolist()
accuracy = sum(t==p for t,p in zip(true_labels,predicted_labels))/len(true_labels)
mp,mr,mf,_ = precision_recall_fscore_support(true_labels,predicted_labels,average='macro',zero_division=0); _,_,wf,_ = precision_recall_fscore_support(true_labels,predicted_labels,average='weighted',zero_division=0)
print('Test Accuracy:', f'{accuracy:.2%}'); print('Macro Precision:', f'{mp:.4f}'); print('Macro Recall:', f'{mr:.4f}'); print('Macro F1:', f'{mf:.4f}'); print('Weighted F1:', f'{wf:.4f}'); print(confusion_matrix(true_labels,predicted_labels))